# Brain Tumor Classification - Train All Models (Colab)

This notebook trains **every model** in the project on the Figshare brain tumor dataset:

**End-to-end fine-tuned CNNs**
- `cnn_baseline`
- `googlenet`
- `resnet18`
- `vgg19`

**Pretrained CNN feature extraction + SVM (transfer learning)**
- `googlenet+svm`
- `resnet18+svm`
- `vgg19+svm`

All models use 5-fold cross-validation (patient leakage free) and results are written to
`reports/<model>_cv_metrics.json`. At the end a combined table + chart is produced.

**Before running:** set `Runtime -> Change runtime type -> GPU` (T4 is fine).

## 1. Check the runtime

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected - training will be very slow.")

## 2. Clone the repository

Set `BRANCH` in the next cell to your working branch (or leave `"main"`).

If the repo is private, add a Colab secret named `PAT` (a GitHub token with repo read access).

In [ ]:
import os

REPO_URL = "https://github.com/Hirujan-R/Brain-Tumor-CNN-Classification.git"
REPO_DIR = "/content/Brain-Tumor-CNN-Classification"

# Set this to your branch name, e.g. "feature/train-all-models".
# Leave as "main" (or None) to use the repository default branch.
BRANCH = "main"

if not os.path.isdir(REPO_DIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("PAT")
    except Exception:
        pass
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    !git clone {url} {REPO_DIR}

%cd /content
os.chdir(REPO_DIR)

if BRANCH:
    !git fetch --all --quiet
    !git checkout {BRANCH}
    !git pull --quiet || true

print("Current branch:")
!git branch --show-current
!git log --oneline -1
!ls

## 3. Install dependencies

We only install the packages that Colab does not already ship, to avoid clobbering the
CUDA-enabled `torch`/`torchvision` that Colab provides.

In [ ]:
%cd /content/Brain-Tumor-CNN-Classification
!pip install -q "dvc[s3]" mlflow pytorch-grad-cam h5py joblib scipy seaborn opencv-python pyyaml tqdm boto3
print("done")

## 4. Configure AWS credentials and pull the DVC data

Add Colab secrets named `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`.
If the data is already present (e.g. from Google Drive) this step is skipped automatically.

In [ ]:
import os

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
try:
    from google.colab import userdata
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION"]:
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass
except Exception as exc:
    print("Colab secrets not available:", exc)

%cd /content/Brain-Tumor-CNN-Classification
if not os.path.exists("data/processed/processed_index.csv"):
    !dvc pull -q || echo "dvc pull failed - check that the AWS secrets are set correctly"
else:
    print("Processed data already present.")

## 5. Sanity-check the dataset

In [ ]:
import os
import pandas as pd

index_path = "data/processed/processed_index.csv"
index = pd.read_csv(index_path)
print("samples:", len(index))
print("patients:", index["patient_id"].nunique())
print("classes (original labels):", index["label"].value_counts().sort_index().to_dict())
print("folds:", index["fold"].value_counts().sort_index().to_dict())
missing = [p for p in index["processed_path"] if not os.path.exists(p)]
print("missing files:", len(missing))
assert len(missing) == 0, "Some processed .npy files are missing"

## 6. Training configuration

Adjust `EPOCHS` to trade off runtime vs. accuracy. SVM models are fast; the CNNs take
the bulk of the time (VGG19 is the heaviest).

In [ ]:
import sys

FOLDS = 5
BATCH_SIZE = 32
NUM_WORKERS = 2
SEED = 42

FINE_TUNE_MODELS = [
    ("cnn_baseline", 30),
    ("googlenet", 30),
    ("resnet18", 30),
    ("vgg19", 20),
]

TRANSFER_SVM_MODELS = ["googlenet+svm", "resnet18+svm", "vgg19+svm"]

# Set to True to also log runs to MLflow (local ./mlruns by default).
USE_MLFLOW = False
MLFLOW_FLAG = "--mlflow" if USE_MLFLOW else ""

EXTRA = f"--batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} --seed {SEED}"

print("Fine-tune models:", FINE_TUNE_MODELS)
print("Transfer + SVM models:", TRANSFER_SVM_MODELS)
print("Folds:", FOLDS, "| workers:", NUM_WORKERS)

## 7. Train the fine-tuned CNN models

In [ ]:
import time

for model_name, epochs in FINE_TUNE_MODELS:
    print("\n" + "=" * 70)
    print(f"TRAINING {model_name} ({epochs} epochs, {FOLDS}-fold CV)")
    print("=" * 70)
    start = time.time()
    cmd = (f"{sys.executable} -m src.pipelines.train --model {model_name} "
           f"--epochs {epochs} --folds {FOLDS} {EXTRA} {MLFLOW_FLAG}")
    print(cmd)
    !{cmd}
    print(f"{model_name} finished in {(time.time() - start) / 60:.1f} min")

## 8. Train the pretrained feature extractor + SVM models

In [ ]:
import time

for model_name in TRANSFER_SVM_MODELS:
    print("\n" + "=" * 70)
    print(f"TRAINING {model_name} ({FOLDS}-fold CV feature extraction + SVM)")
    print("=" * 70)
    start = time.time()
    cmd = (f"{sys.executable} -m src.pipelines.train --model {model_name} "
           f"--folds {FOLDS} {EXTRA} {MLFLOW_FLAG}")
    print(cmd)
    !{cmd}
    print(f"{model_name} finished in {(time.time() - start) / 60:.1f} min")

## 9. Combined results table

In [ ]:
import glob
import json
import pandas as pd

rows = []
for path in sorted(glob.glob("reports/*_cv_metrics.json")):
    with open(path) as handle:
        payload = json.load(handle)
    agg = payload["aggregate"]
    rows.append({
        "model": payload["model"],
        "approach": payload["approach"],
        "folds": payload["num_folds"],
        "accuracy": agg["accuracy_mean"],
        "accuracy_std": agg["accuracy_std"],
        "balanced_accuracy": agg["balanced_accuracy_mean"],
        "precision": agg["precision_mean"],
        "recall": agg["recall_mean"],
        "f1": agg["f1_mean"],
        "roc_auc": agg["roc_auc_mean"],
    })

results = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
display(results)
results.to_csv("reports/all_models_metrics.csv", index=False)
print("Saved: reports/all_models_metrics.csv")

## 10. Visual comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(results):
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["#2a6fdb" if a == "fine_tune" else "#e08a1e" for a in results["approach"]]
    y = np.arange(len(results))
    ax.barh(y, results["accuracy"], xerr=results["accuracy_std"], color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(results["model"])
    ax.invert_yaxis()
    ax.set_xlabel("Accuracy")
    ax.set_title("5-fold CV accuracy by model")
    ax.set_xlim(0, 1)
    for i, v in enumerate(results["accuracy"]):
        ax.text(v + 0.01, i, f"{v:.3f}", va="center")
    plt.tight_layout()
    plt.savefig("reports/all_models_accuracy.png", dpi=150)
    plt.show()
else:
    print("No results yet - run the training cells above.")

## 11. Per-class report and confusion matrix (best model)

Loads the pooled out-of-fold confusion matrix saved for each model and prints the
classification report for the best-performing one.

In [ ]:
import glob
import json
import numpy as np
import pandas as pd

payloads = {}
for path in glob.glob("reports/*_cv_metrics.json"):
    with open(path) as handle:
        payload = json.load(handle)
    payloads[payload["model"]] = payload

if payloads:
    best_model = max(payloads, key=lambda m: payloads[m]["aggregate"]["accuracy_mean"])
    payload = payloads[best_model]
    print("Best model by mean accuracy:", best_model)
    cm = pd.DataFrame(
        np.array(payload["confusion_matrix"]),
        index=["true_glioma", "true_meningioma", "true_pituitary"],
        columns=["pred_glioma", "pred_meningioma", "pred_pituitary"],
    )
    display(cm)
    display(pd.DataFrame(payload["classification_report"]).T)
else:
    print("No metrics found.")

## 12. Grad-CAM explainability (optional)

Uses the best GoogLeNet fold checkpoint if available, otherwise the committed
`model/model.pth`.

In [ ]:
import os
import numpy as np

from src.inference.predict import load_model, predict
from src.inference.gradcam import generate_gradcam, visualize_gradcam

sample_id = "000896"
image = np.load(f"data/processed/images/{sample_id}.npy")

candidates = [
    "reports/checkpoints/googlenet_fold0/best_model.pth",
    "model/model.pth",
]
model_path = next((p for p in candidates if os.path.exists(p)), None)
print("Using checkpoint:", model_path)

model = load_model(model_path, arch="googlenet", device="cpu")
result = predict(model, image, device="cpu")
print("Prediction:", result["class_name"], "| probabilities:", np.round(result["probabilities"], 3))

cam, probs, pred_class = generate_gradcam(model, image, arch="googlenet")
os.makedirs("output", exist_ok=True)
visualize_gradcam(image, cam, save_path=f"output/gradcam_{sample_id}.png")

## 13. Download the results

Run the cell below to zip the metrics for reporting.

In [ ]:
import shutil

shutil.make_archive("/content/brain_tumor_results", "zip", "reports")
print("Created /content/brain_tumor_results.zip")
try:
    from google.colab import files
    files.download("/content/brain_tumor_results.zip")
except Exception as exc:
    print("Download the file manually from the Colab file browser:", exc)

## 14. Report back

After the run, share the contents of `reports/all_models_metrics.csv` (the combined table).
The metric columns are:
`accuracy, balanced_accuracy, precision, recall, f1, roc_auc` (mean over 5 folds).